In [1]:
import os

if os.getcwd().endswith('notebooks'):
    os.chdir('..')
print(os.getcwd())

/home/cmcoutosilva/Projects/gitlab/sqldeps


In [2]:
import json
from sqldeps.llm_parsers import GroqExtractor, OpenaiExtractor, DeepseekExtractor

In [3]:
def read_sql(sql_file, include_test=True):
    with open(sql_file) as file:
        sql_content = file.read()
    if not include_test:
        return sql_content
    else:
        test_json_path = (
            sql_file
            .replace('/test_sql/', '/test_sql_outputs/')
            .replace('.sql', '_expected.json')
        )
        return sql_content, test_json_path
    
def compute_test_success(result, test_json_path) -> bool:
    with open(test_json_path) as json_file:
        expected_result = json.load(json_file)
    return result.to_dict() == expected_result

def print_section(title: str, width: int = 80, char: str = '=') -> None:
    print(f"{' ' + title + ' ':{char}^{width}}")


def run_pipeline(extractor, test_sql_path):
    """Run the pipeline and print results in a formatted way."""
    sql_content, test_json_path = read_sql(test_sql_path)
    
    # Print SQL section
    print_section('Source Query')
    print(sql_content, end='\n\n')

    # Run extractor
    result = extractor.extract_from_query(sql_content)

    # Print response section
    print_section("Model Response")
    print(extractor.last_response)
    print()

    # Compute & print unit test
    if compute_test_success(result,test_json_path):
        print('\033[92m→ Test passed!\033[0m')
    else:
        print('\033[91m→ Test did not pass!\033[0m')
    print('='*80)

    return result

# Examples

In [4]:
model_options = dict(
    openai = [
        'GPT-4o',
        'GPT-4o-mini'
    ],
    deepseek = [
        'deepseek-chat',
        # 'deepseek-reasoner'
    ],
    groq = [
        # see all options here: https://console.groq.com/docs/models
        'gemma2-9b-it',            # Google
        'llama-3.3-70b-versatile', # Meta
        'llama-3.1-8b-instant',    # Meta
        'llama3-70b-8192',         # Meta  
        'llama3-8b-8192',          # Meta
        'mixtral-8x7b-32768'       # Mistral
    ]
)

In [5]:
# Params
framework = GroqExtractor
target_model = 'llama-3.3-70b-versatile'
prompt_path = 'sqldeps/configs/prompts/default.yml'

# Instantiate extractor
extractor = framework(
    model=target_model,
    params={'temperature':0},
    prompt_path=prompt_path
)

In [6]:
# Simple query selecting a subset of columns
run_pipeline(extractor, 'tests/test_sql/example1.sql')

================================= Source Query =================================
-- Simple query selecting a subset of columns
SELECT id, name FROM users

================================ Model Response ================================
{
   "dependencies": {"users": ["id", "name"]},
   "outcomes": {}
}

→ Test passed!


SQLAnalysis(dependencies={'users': ['id', 'name']}, outcomes={})

In [7]:
# Simple query selecting all columns
run_pipeline(extractor, 'tests/test_sql/example2.sql')

================================= Source Query =================================
-- Simple query selecting all columns
SELECT * FROM users LIMIT 100

================================ Model Response ================================
{
   "dependencies": {"users": ["*"]},
   "outcomes": {}
}

→ Test passed!


SQLAnalysis(dependencies={'users': ['*']}, outcomes={})

In [8]:
# Query with table alias, with and without database specification, and join
run_pipeline(extractor, 'tests/test_sql/example3.sql')

================================= Source Query =================================
-- Query with table alias, with and without database specification, and join
SELECT u.id, u.name, o.order_id
FROM my_db.users u
JOIN orders AS o ON u.id = o.user_id

================================ Model Response ================================
{
   "dependencies": {
      "my_db.users": ["id", "name"],
      "orders": ["order_id", "user_id"]
   },
   "outcomes": {}
}

→ Test passed!


SQLAnalysis(dependencies={'my_db.users': ['id', 'name'], 'orders': ['order_id', 'user_id']}, outcomes={})

In [9]:
# Query with table alias, with and without database specification, and join, and where clauses
run_pipeline(extractor, 'tests/test_sql/example4.sql')

================================= Source Query =================================
-- Query with table alias, with and without database specification, and join, and where clauses
SELECT u.id, u.name, o.order_id
FROM my_db.users u
JOIN orders AS o ON u.id = o.user_id
WHERE u.status = 'active'
    AND o.order_date >= '2024-01-01'
    AND o.total_amount > 100.00
    AND u.email LIKE '%@company.com'
    AND o.order_type IN ('retail', 'wholesale')
    AND (
        o.shipping_status = 'pending'
        OR (o.shipping_status = 'processed' AND o.priority_level = 'high')
    );

================================ Model Response ================================
{
   "dependencies": {
      "my_db.users": ["id", "name", "status", "email"],
      "orders": ["order_id", "user_id", "order_date", "total_amount", "order_type", "shipping_status", "priority_level"]
   },
   "outcomes": {}
}

→ Test passed!


SQLAnalysis(dependencies={'my_db.users': ['email', 'id', 'name', 'status'], 'orders': ['order_date', 'order_id', 'order_type', 'priority_level', 'shipping_status', 'total_amount', 'user_id']}, outcomes={})

In [10]:
# Simple CTE
run_pipeline(extractor, 'tests/test_sql/example5.sql')

================================= Source Query =================================
-- Simple CTE
WITH user_orders AS (
    SELECT user_id, COUNT(*) as order_count
    FROM orders
    GROUP BY user_id
)
SELECT u.name, uo.order_count
FROM users u
JOIN user_orders uo ON u.id = uo.user_id;

================================ Model Response ================================
{
   "dependencies": {
      "orders": ["user_id"],
      "users": ["id", "name"]
   },
   "outcomes": {}
}

→ Test passed!


SQLAnalysis(dependencies={'orders': ['user_id'], 'users': ['id', 'name']}, outcomes={})

In [11]:
# Simple Subquery 1
run_pipeline(extractor, 'tests/test_sql/example6.sql')

================================= Source Query =================================
-- Simple Subquery 1
SELECT 
    u.name, 
    (
        SELECT COUNT(*) 
        FROM orders o 
        WHERE o.user_id = u.id
        GROUP BY o.user_id
    ) as order_count
FROM users u;

================================ Model Response ================================
{
   "dependencies": {
      "users": ["name", "id"],
      "orders": ["user_id"]
   },
   "outcomes": {}
}

→ Test passed!


SQLAnalysis(dependencies={'orders': ['user_id'], 'users': ['id', 'name']}, outcomes={})

In [12]:
# Simple Subquery 2
run_pipeline(extractor, 'tests/test_sql/example7.sql')

================================= Source Query =================================
-- Simple Subquery 2
SELECT 
    u.name, 
    uo.order_count
FROM users u
JOIN (
    SELECT 
        user_id, 
        COUNT(*) as order_count
    FROM orders
    GROUP BY user_id
) uo ON u.id = uo.user_id;

================================ Model Response ================================
{
   "dependencies": {
      "users": ["id", "name"],
      "orders": ["user_id"]
   },
   "outcomes": {}
}

→ Test passed!


SQLAnalysis(dependencies={'orders': ['user_id'], 'users': ['id', 'name']}, outcomes={})

In [13]:
# Postgres Function
run_pipeline(extractor, 'tests/test_sql/example8.sql')

================================= Source Query =================================
-- Postgres Function
CREATE OR REPLACE FUNCTION web_import."build_Api_Property_Defor"()
 RETURNS void
 LANGUAGE plpgsql
AS $function$BEGIN
	TRUNCATE TABLE web_import."Api_Property_Defor";

INSERT INTO web_import."Api_Property_Defor"(
		"PropertyId", "Year", "Ha"
	)
	SELECT ps."PropertyId", d."Year", avg("Defor") AS "Ha"
FROM build_public."Property_Shape" ps
INNER JOIN (
				SELECT "ShapeId", "Year"::INTEGER, SUM("areaha") AS "Defor"
FROM build_spatial."Shape_Defor"
WHERE "Year"::text ~ '^[0-9]+$'
-- and "areaha">6.25
GROUP BY "ShapeId", "Year"::INTEGER
			) d
				ON
d."ShapeId" = ps."ShapeId"
WHERE ps."PropertyId" IS NOT NULL
GROUP BY ps."PropertyId", d."Year";

END
$function$

================================ Model Response ================================
{
   "dependencies": {
      "build_public.Property_Shape": ["PropertyId", "ShapeId"],
      "build_spatial.Shape_Defor": ["ShapeId", "Year", "areaha"]

SQLAnalysis(dependencies={'build_public.Property_Shape': ['PropertyId', 'ShapeId'], 'build_spatial.Shape_Defor': ['ShapeId', 'Year', 'areaha'], 'web_import.Api_Property_Defor': ['Ha', 'PropertyId', 'Year']}, outcomes={'web_import.Api_Property_Defor': ['Ha', 'PropertyId', 'Year']})

In [14]:
# Multiple queries with CTEs & function
run_pipeline(extractor, 'tests/test_sql/example9.sql')

================================= Source Query =================================
-- Multiple queries with CTEs & function
CREATE OR REPLACE FUNCTION make_pgi_shape_geom_clusters()
  RETURNS VOID
  LANGUAGE plpgsql
AS $function$
BEGIN

    -- Build table with cluster + geom data
    DROP TABLE IF EXISTS pgi_shape_geom_clusters CASCADE;
    CREATE TABLE pgi_shape_geom_clusters AS
        SELECT
            pgic."PropertyGroupId",
            pgic."ShapeGroupId",
            sh.geom,
            pgic."ShapeCluster" 
        FROM
            pgi_shape_clusters pgic
        LEFT JOIN
            spatial."Shape" sh
        ON
            pgic."PropertyGroupId" = sh."ShapeId";

    -- Integrity check: A Property observation should have at most one row
    ALTER TABLE pgi_shape_geom_clusters ADD PRIMARY KEY ("PropertyGroupId","ShapeGroupId");
    ANALYZE VERBOSE pgi_shape_geom_clusters;

END
$function$;

WITH user_orders AS (
    SELECT user_id, COUNT(*) as order_count
    FROM orders
    GROU

SQLAnalysis(dependencies={'orders': ['user_id'], 'pgi_shape_clusters': ['PropertyGroupId', 'ShapeCluster', 'ShapeGroupId'], 'spatial.Shape': ['ShapeId', 'geom'], 'users': ['id', 'name']}, outcomes={'pgi_shape_geom_clusters': ['PropertyGroupId', 'ShapeCluster', 'ShapeGroupId', 'geom']})

In [15]:
# Complex query
run_pipeline(extractor, 'tests/test_sql/example10.sql')

================================= Source Query =================================
-- PostgreSQL function that uses CTEs and creates a table
CREATE OR REPLACE FUNCTION generate_sales_report()
RETURNS void AS $$
BEGIN
    -- Use CTEs to process data
    WITH cte_sales AS (
        SELECT 
            s.id AS sale_id, 
            s.amount, 
            c.customer_name 
        FROM sales s
        JOIN customers c ON s.customer_id = c.id
    ),
    cte_products AS (
        SELECT 
            p.product_id, 
            p.product_name 
        FROM products p
    )
    -- Insert the processed data into a new table
    INSERT INTO reports.sales_report (sale_id, customer_name, product_name)
    SELECT 
        cte_sales.sale_id, 
        cte_sales.customer_name, 
        cte_products.product_name
    FROM cte_sales
    JOIN cte_products ON cte_sales.sale_id = cte_products.product_id;
END;
$$ LANGUAGE plpgsql;

-- Truncate a table
TRUNCATE TABLE logs;

-- Query from a specific database
SELEC

SQLAnalysis(dependencies={'customers': ['customer_name', 'id'], 'employees': ['*'], 'logs': [], 'my_db.orders': ['order_date', 'order_id', 'total_amount'], 'products': ['product_id', 'product_name'], 'reports.sales_report': ['customer_name', 'product_name', 'sale_id'], 'sales': ['amount', 'customer_id', 'id']}, outcomes={'logs': [], 'reports.sales_report': ['customer_name', 'product_name', 'sale_id']})